In [0]:
class invoiceStream():

    def __init__(self):
        self.base_data_dir = "/Volumes/dev/spark_db/volume-2/invoices/"

    def getSchema(self):
        return """
            InvoiceNumber string,
            CreatedTime bigint,
            StoreID string,
            PosID string,
            CashierID string,
            CustomerType string,
            CustomerCardNo string,
            TotalAmount double,
            NumberOfItems bigint,
            PaymentMethod string,
            TaxableAmount double,
            CGST double,
            SGST double,
            CESS double,
            DeliveryType string,
            DeliveryAddress struct<
                AddressLine string,
                City string,
                ContactNumber string,
                PinCode string,
                State string
            >,
            InvoiceLineItems array<
                struct<
                    ItemCode string,
                    ItemDescription string,
                    ItemPrice double,
                    ItemQty bigint,
                    TotalValue double
                >
            >
        """

    def readInvoices(self):
        return (
            spark.readStream
                .format("json")
                .schema(self.getSchema())
                .load(f"{self.base_data_dir}/data/invoices")
        )

    def explodeInvoices(self, invoiceDF):
       return invoiceDF.select(
        col("InvoiceNumber"),
        col("CreatedTime"),
        col("StoreID"),
        col("PosID"),
        col("CustomerType"),
        col("PaymentMethod"),
        col("DeliveryType"),
        col("DeliveryAddress.City").alias("City"),
        col("DeliveryAddress.State").alias("State"),
        col("DeliveryAddress.PinCode").alias("PinCode"),
        explode(col("InvoiceLineItems")).alias("LineItem")
        )

    def flattenInvoices(self, explodedDF):
        from pyspark.sql.functions import expr

        return (
            explodedDF
                .withColumn(
                    "ItemCode", 
                    expr("LineItem.ItemCode")
                )
                .withColumn(
                    "ItemDescription", 
                    expr("LineItem.ItemDescription")
                )
                .withColumn(
                    "ItemPrice", 
                    expr("LineItem.ItemPrice")
                )
                .withColumn(
                    "ItemQty", 
                    expr("LineItem.ItemQty")
                )
                .withColumn(
                    "TotalValue", 
                    expr("LineItem.TotalValue")
                )
                .drop("LineItem")
        )

    def appendInvoices(self, flattenedDF):
        return (
            flattenedDF.writeStream
                .format("delta")
                .option(
                    "checkpointLocation",
                    f"{self.base_data_dir}/chekpoint/invoices"
                )
                .outputMode("append")
                .toTable("invoice_line_items")
        )

    def process(self):
        print("Starting Invoice Processing Stream...", end="")

        invoicesDF = self.readInvoices()

        explodedDF = self.explodeInvoices(invoicesDF)

        resultDF = self.flattenInvoices(explodedDF)

        sQuery = self.appendInvoices(resultDF)

        print("Done\n")

        return sQuery